# Lab 7 — The shape of data: three ways to fill a cloud

**Accompanies Lecture 7 (when spaces are "the same", and how to build a space from points).**

You measure what distances cannot see, put numbers on how dense a sample must be, and then build the three complexes of the lecture (Čech, Rips, Alpha) yourself and watch them agree and disagree.

| # | Exercise | What you build | When |
|---|---|---|---|
| 1 | Distances against topology | `pairwise_distances`, `w1_distance_distributions`, `hausdorff` | session, 10 min |
| 2 | Gaps in random samples | `largest_gap`, `prob_gap_exceeds`, `exact_prob_gap_exceeds`, `smallest_N` | session, 15 min |
| 3 | The three rules on four small clouds | `cech_triangle_value`, `rips_triangle_value`, `alpha_complex`, `loop_intervals` | session, 25–30 min |
| 4 | Delaunay and the cost of a complex | `empty_circle_violations`, `euler_characteristic`, `delaunay_sizes_2d`, `delaunay_sizes_3d`, `count_by_dimension` | session, 15 min |
| 5 | Homotopy and knots (optional) | `straight_line_homotopy`, `min_far_distance`, `alexander_determinant`, `find_collisions` | take-home, 15 min |

Work in pairs: one person writes, the other explains what the code must do; swap after Exercise 2. Every exercise ends with an `assert` cell: if your code is correct, the notebook runs from top to bottom without an error. Cells marked **GIVEN** are complete; cells with `# TODO` are yours. The seeds are fixed; change `SEED` below to check that your solutions do not depend on luck.

You need `ripser` and `gudhi` (the next cell installs them if they are missing).

In [ ]:
try:
    import ripser, gudhi
except ImportError:
    %pip install -q ripser gudhi
    import ripser, gudhi
import warnings, itertools, time
from math import comb
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay

warnings.filterwarnings("ignore")
NAVY, TEAL, AMBER, MUTE = "#1E2761", "#1C7293", "#E8871E", "#6E7691"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 0
print("numpy", np.__version__, "| gudhi", gudhi.__version__)

## Exercise 1 — Distances against topology

A circle with a small gap is topologically an interval; a stretched circle is still a circle. What do distances say? (`curve_points` gives N points equally spaced along a circle; a gap removes the points that fall in it, which is what a real sample of an arc looks like.)

You write:

1. `pairwise_distances(P)`: the sorted array of all distances between pairs of points of `P` (each pair once).
2. `w1_distance_distributions(P, Q, levels=4000)`: the Wasserstein-1 distance between the *distributions of pairwise distances* of two point sets. The sets may differ in size (the gap removes points), so compare the two sorted arrays as quantile functions: for `levels` equally spaced levels $u=(i+\tfrac12)/\text{levels}$ read off each sorted array at position $u$ (`np.interp` against $(j+\tfrac12)/\text{len}$) and average the absolute differences.
3. `hausdorff(P, Q)`: the Hausdorff distance, the larger of the two directed distances $\max_{p\in P}\min_{q\in Q}|p-q|$ and $\max_{q\in Q}\min_{p\in P}|p-q|$.

In [ ]:
def curve_points(N, gap=0.0, stretch=1.0):                      # GIVEN: N points equally spaced along a (stretched) circle; the points that fall in a gap are removed
    M = 8000; a = 2 * np.pi * np.arange(M + 1) / M
    X = np.column_stack([stretch * np.cos(a), np.sin(a) / stretch])
    cum = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(X, axis=0), axis=1))])
    target = np.arange(N) / N * cum[-1]
    pts = np.column_stack([np.interp(target, cum, X[:, 0]), np.interp(target, cum, X[:, 1])]); ang = np.interp(target, cum, a)
    return pts[np.abs((ang + np.pi) % (2 * np.pi) - np.pi) >= gap / 2]            # a gap of the given width, centred at angle 0

def pairwise_distances(P):
    """Sorted array of the distances between all pairs of points (each pair once)."""
    # TODO
    raise NotImplementedError

def w1_distance_distributions(P, Q, levels=4000):
    """W1 between the distributions of pairwise distances of two point sets (the sets may differ in size)."""
    # TODO
    raise NotImplementedError

def hausdorff(P, Q):
    """Hausdorff distance between two finite point sets."""
    # TODO
    raise NotImplementedError

**Part A.** The reference circle, a circle with a gap of 0.5 rad, and a circle stretched by 1.5 (1000 points before the gap is cut).

In [ ]:
# GIVEN
N = 1000; ref = curve_points(N)
shapes = {"gap 0.5 rad": curve_points(N, gap=0.5), "stretch 1.5": curve_points(N, stretch=1.5)}
res1 = {k: (w1_distance_distributions(ref, S), hausdorff(ref, S)) for k, S in shapes.items()}
for k, (w, h) in res1.items(): print(f"{k:12s}: W1 of the distance distributions {w:.4f}, Hausdorff distance {h:.3f}")
fig, ax = plt.subplots(1, 3, figsize=(10, 2.6))
for a, (t, S, c) in zip(ax, (("circle", ref, TEAL), ("gap 0.5 rad", shapes["gap 0.5 rad"], AMBER), ("stretch 1.5", shapes["stretch 1.5"], TEAL))): a.scatter(*S.T, s=4, color=c); a.set_title(t, fontsize=10); a.set_aspect("equal"); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part A
(wg, hg), (ws, hs) = res1["gap 0.5 rad"], res1["stretch 1.5"]
assert wg < ws / 5, "by the distances, the gap is a much smaller change than the stretch"
assert abs(hg - 0.249) < 0.03 and abs(hs - 0.500) < 0.03, "Hausdorff distances: about 0.25 for the gap, 0.50 for the stretch"
assert abs(wg - 0.0088) < 0.004 and abs(ws - 0.132) < 0.02
print("Exercise 1, Part A passed: the topology changed (a circle was cut) where the distances hardly moved.")

**Part B.** Make the gap smaller and smaller. The distance changes smoothly towards zero. What does the topology do?

In [ ]:
# GIVEN
gaps = [0.02, 0.05, 0.1, 0.25, 0.5, 1.0]
w_gap = [w1_distance_distributions(ref, curve_points(N, gap=g)) for g in gaps]
print("gap (rad):", gaps); print("W1:       ", [round(w, 5) for w in w_gap])
fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(gaps, w_gap, "o-", color=AMBER); ax.set_xlabel("gap (rad)"); ax.set_ylabel("W1 of the distance distributions"); plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part B
assert all(np.diff(w_gap) > 0), "the distance grows smoothly with the gap"
assert w_gap[0] < 0.0005, "a gap of 0.02 rad (about one degree) is nearly invisible to the distances, but the loop is already cut"
print("Exercise 1 passed.")

*Discuss:* for a gap of 0.02 rad the distances hardly change and the shape is no longer a circle. What kind of quantity could jump at the moment the gap opens?

## Exercise 2 — Gaps in random samples

A random sample of a full circle can contain a large empty stretch. When that happens, we cannot tell the sample from one taken from an arc.

You write:

1. `largest_gap(angles)`: the largest gap between consecutive points on the circle, **including the gap that wraps around** from the last point to the first.
2. `prob_gap_exceeds(N, g, trials, rng)`: the fraction of `trials` random samples of $N$ uniform angles whose largest gap exceeds `g`.
3. `exact_prob_gap_exceeds(N, g)`: the exact value, $P=\sum_{k\ge 1}(-1)^{k+1}\binom{N}{k}\big(1-k\,g/2\pi\big)_+^{N-1}$, where $(x)_+=\max(x,0)$ and the sum runs while $kg<2\pi$.
4. `smallest_N(g, target=0.05)`: the smallest $N$ (start at 5) for which the exact probability is below `target`.

In [ ]:
def largest_gap(angles):
    """Largest gap between consecutive points on the circle, including the wrap-around gap."""
    # TODO
    raise NotImplementedError

def prob_gap_exceeds(N, g, trials, rng):
    """Fraction of random samples of N points on the circle with a gap larger than g."""
    # TODO
    raise NotImplementedError

def exact_prob_gap_exceeds(N, g):
    """Exact probability that the largest gap of N uniform points on the circle exceeds g."""
    # TODO
    raise NotImplementedError

def smallest_N(g, target=0.05):
    """Smallest N (from 5 on) such that P(largest gap > g) < target."""
    # TODO
    raise NotImplementedError

In [ ]:
# CHECK: the unit test for the wrap-around gap, and the exact values
squeezed = np.linspace(0, 1, 10)                      # all points in [0, 1]: the empty arc is 2 pi - 1 long, and it wraps around
assert abs(largest_gap(squeezed) - (2 * np.pi - 1)) < 1e-9, "do not forget the gap between the last point and the first"
assert abs(largest_gap([0.0, np.pi]) - np.pi) < 1e-12
ex = {n: exact_prob_gap_exceeds(n, 0.5) for n in (20, 30, 50, 100, 200)}
print({n: round(100 * p, 2) for n, p in ex.items()}, "(percent)")
assert ex[20] > 0.9999 and abs(ex[30] - 0.9909) < 5e-4 and abs(ex[50] - 0.6381) < 5e-4 and abs(ex[100] - 0.0270) < 5e-4 and 0 < ex[200] < 1e-4
print("Exercise 2, Part A passed.")

**Part B.** Check the formula against a simulation (20000 random samples for each $N$).

In [ ]:
# GIVEN
rng = np.random.default_rng(SEED + 1)
sim = {n: prob_gap_exceeds(n, 0.5, 20000, rng) for n in (30, 50, 100)}
for n, s in sim.items(): print(f"N = {n:3d}: simulation {100 * s:6.2f}%   exact {100 * ex[n]:6.2f}%")

In [ ]:
# CHECK: Part B
for n, s in sim.items(): assert abs(s - ex[n]) < 0.02, f"simulation and formula disagree at N = {n}"
print("Exercise 2, Part B passed.")

**Part C.** How many points do we need for a 5% chance of a gap larger than $g$?

In [ ]:
# GIVEN
need = {g: smallest_N(g) for g in (1.0, 0.5, 0.25)}
print("smallest N with P(gap > g) < 5%:", need)

In [ ]:
# CHECK: Part C
assert need == {1.0: 40, 0.5: 92, 0.25: 206}, need
assert need[0.25] > 2 * need[0.5] > 4 * need[1.0] - 10, "halving the gap a bit more than doubles the number of points"
print("Exercise 2 passed.")

*Discuss:* to resolve a feature of size $g$ we need about $N \propto (1/g)\log(1/g)$ points. What does this say about a sample that has to resolve both a big loop and a narrow channel?

## Exercise 3 — The three rules on four small clouds

The same points, the same radius $r$ (the radius of the balls), three rules for which sets of points form a simplex:

* **Čech(r):** the balls of radius $r$ around the points have a common point, i.e. the smallest enclosing circle has radius at most $r$.
* **Rips(r):** all pairwise distances are at most $2r$.
* **Alpha(r):** the balls cut along Voronoi cells have a common point. For a Delaunay triangle this is the circumradius; for an edge it is **half its length if the circle with that edge as diameter contains no other point of the cloud** (the Gabriel rule), and otherwise the smaller circumradius of the triangles it belongs to.

Each simplex has a **value**: the radius at which it enters. The edges of Čech and Rips both enter at half their length. You write:

1. `cech_triangle_value(a, b, c)`: the radius of the smallest circle containing the three points: half the longest side if the triangle is right or obtuse (longest side$^2\ge$ sum of the other two squared), and the circumradius otherwise.
2. `rips_triangle_value(a, b, c)`: half the longest side.
3. `alpha_complex(P, gabriel=True)`: using `scipy.spatial.Delaunay`, return `(edge_vals, tri_vals)`, dictionaries from sorted index tuples to values (triangle: circumradius; edge: see the Gabriel rule; with `gabriel=False` every edge simply gets half its length, which is the mistake of Part C).
4. `loop_intervals(n, edge_vals, tri_vals, rmax)`: the list of intervals $[r_1,r_2)$ on which the complex has at least one loop ($\beta_1\ge 1$). The complex only changes at the values of its simplices, so sort those values and test one radius **in the middle** of each stretch between two consecutive values; merge consecutive stretches with a loop. The helper `betti01_at(n, edge_vals, tri_vals, r)` (given) returns $(\beta_0,\beta_1)$ of the complex of all simplices with value at most `r`.

In [ ]:
def circumradius(a, b, c):                                       # GIVEN
    A, B, C = np.linalg.norm(b - c), np.linalg.norm(a - c), np.linalg.norm(a - b)
    area2 = abs((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))
    return np.inf if area2 < 1e-14 else A * B * C / (2 * area2)

def betti01_at(n, edge_vals, tri_vals, r, eps=1e-12):            # GIVEN: b0 and b1 of the complex of all simplices with value <= r (b1 over GF(2))
    E = [e for e, v in edge_vals.items() if v <= r + eps]; idx = {e: k for k, e in enumerate(E)}; parent = list(range(n))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    comps = n
    for a, b in E:
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb; comps -= 1
    low = {}; rank = 0
    for (a, b, c), v in tri_vals.items():
        if v > r + eps: continue
        try: col = sorted([idx[(a, b)], idx[(b, c)], idx[(a, c)]])
        except KeyError: continue
        while col:
            l = col[-1]
            if l not in low: low[l] = col; rank += 1; break
            col = sorted(set(col) ^ set(low[l]))
    return comps, len(E) - n + comps - rank

def half_edges(P):                                               # GIVEN: the Cech and Rips edges: half the distance
    n = len(P); return {(i, j): float(np.linalg.norm(P[i] - P[j]) / 2) for i in range(n) for j in range(i + 1, n)}

def cech_triangle_value(a, b, c):
    """Radius of the smallest circle containing three points."""
    # TODO
    raise NotImplementedError

def rips_triangle_value(a, b, c):
    """Half the longest side of the triangle."""
    # TODO
    raise NotImplementedError

def alpha_complex(P, gabriel=True):
    """(edge_vals, tri_vals) of the Alpha complex of the points P, from the Delaunay triangulation."""
    # TODO
    raise NotImplementedError

def loop_intervals(n, edge_vals, tri_vals, rmax):
    """Intervals [r1, r2) of the radius on which the complex has at least one loop."""
    # TODO
    raise NotImplementedError

In [ ]:
# GIVEN: the three complexes of a cloud, and four small clouds with the answers we expect
def cech_complex(P):
    n = len(P); return half_edges(P), {t: cech_triangle_value(*P[list(t)]) for t in itertools.combinations(range(n), 3)}
def rips_complex(P):
    n = len(P); return half_edges(P), {t: rips_triangle_value(*P[list(t)]) for t in itertools.combinations(range(n), 3)}

clouds = {"equilateral triangle": np.array([[0, 0], [1, 0], [0.5, np.sqrt(3) / 2]]), "obtuse triangle": np.array([[0, 0], [1, 0], [0.5, 0.1]]),
          "unit square": np.array([[0, 0], [1, 0], [1, 1], [0, 1]], float), "regular hexagon": np.array([[np.cos(k * np.pi / 3), np.sin(k * np.pi / 3)] for k in range(6)])}
expected = {"equilateral triangle": {"cech": [[0.5, 0.5774]], "rips": [], "alpha": [[0.5, 0.5774]]}, "obtuse triangle": {"cech": [], "rips": [], "alpha": []},
            "unit square": {"cech": [[0.5, 0.7071]], "rips": [[0.5, 0.7071]], "alpha": [[0.5, 0.7071]]}, "regular hexagon": {"cech": [[0.5, 1.0]], "rips": [[0.5, 0.8660]], "alpha": [[0.5, 1.0]]}}
found = {}
for name, P in clouds.items():
    n = len(P); rmax = 2.0
    found[name] = {"cech": loop_intervals(n, *cech_complex(P), rmax), "rips": loop_intervals(n, *rips_complex(P), rmax), "alpha": loop_intervals(n, *alpha_complex(P), rmax)}
    print(f"{name:22s}", {k: np.round(v, 4).tolist() for k, v in found[name].items()})

In [ ]:
# CHECK: Part A
for name in clouds:
    for rule in ("cech", "rips", "alpha"):
        got, want = found[name][rule], expected[name][rule]
        assert len(got) == len(want) and all(abs(g - w) < 1e-3 for gi, wi in zip(got, want) for g, w in zip(gi, wi)), f"{name}, {rule}: got {np.round(got, 4).tolist()}, expected {want}"
print("Exercise 3, Part A passed: Cech and Alpha see the equilateral triangle's loop, Rips never does; in the hexagon Rips closes the loop at 0.866, the others at 1.")

**Part B. Units.** Libraries do not use our radius. `ripser` works with the **edge length** (so $\varepsilon=2r$) and `gudhi`'s alpha complex with the **squared radius** ($r^2$). Convert their output and compare it with yours.

In [ ]:
# GIVEN
from ripser import ripser
def real_intervals(arr, tol=1e-5): return [list(map(float, iv)) for iv in np.asarray(arr).reshape(-1, 2) if np.isfinite(iv[1]) and iv[1] - iv[0] > tol]
unit_check = {}
for name, P in clouds.items():
    dg = ripser(P, maxdim=1)["dgms"][1]; rips_lib = [[a / 2, b / 2] for a, b in real_intervals(dg)]
    st = gudhi.AlphaComplex(points=P + 1e-7 * np.random.default_rng(0).normal(size=P.shape)).create_simplex_tree(); st.persistence()
    alpha_lib = [[np.sqrt(a), np.sqrt(b)] for a, b in real_intervals(st.persistence_intervals_in_dimension(1))]
    unit_check[name] = (rips_lib, alpha_lib); print(f"{name:22s} ripser / 2: {np.round(rips_lib, 4).tolist()}   gudhi sqrt: {np.round(alpha_lib, 4).tolist()}")

In [ ]:
# CHECK: Part B
for name in clouds:
    rips_lib, alpha_lib = unit_check[name]
    for lib, mine in ((rips_lib, found[name]["rips"]), (alpha_lib, found[name]["alpha"])):
        assert len(lib) == len(mine) and all(abs(a - b) < 2e-3 for x, y in zip(lib, mine) for a, b in zip(x, y)), f"{name}: library {lib}, yours {mine}"
print("Exercise 3, Part B passed: your intervals agree with the libraries after the conversion ripser / 2 and sqrt(gudhi).")

**Part C. A trap.** Run your Alpha complex with `gabriel=False` (every edge enters at half its length) on all four clouds. One cloud gives a wrong answer. Which one, and why does only that one expose the mistake?

In [ ]:
# GIVEN
wrong = {name: loop_intervals(len(P), *alpha_complex(P, gabriel=False), 2.0) for name, P in clouds.items()}
for name in clouds: print(f"{name:22s} without the Gabriel rule: {np.round(wrong[name], 4).tolist()}   with it: {np.round(found[name]['alpha'], 4).tolist()}")
print("circumradius of the obtuse triangle:", round(circumradius(*clouds["obtuse triangle"]), 4))

In [ ]:
# CHECK: Part C
assert all(len(wrong[n]) == len(found[n]["alpha"]) for n in ("equilateral triangle", "unit square", "regular hexagon")), "the other three clouds are not affected"
assert len(wrong["obtuse triangle"]) == 1 and abs(wrong["obtuse triangle"][0][0] - 0.5) < 1e-3 and abs(wrong["obtuse triangle"][0][1] - 1.3) < 1e-3, "without the Gabriel rule the obtuse triangle gets a spurious loop [0.5, 1.3)"
assert found["obtuse triangle"]["alpha"] == [], "with the rule there is no loop"
print("Exercise 3, Part C passed: the long edge of an obtuse triangle enters together with the triangle, not at half its length.")

**Part D. A discovery.** The lecture claims that the Čech and Alpha complexes have the same topology (the nerve theorem, next time). On 30 random clouds of 7 points compare their loop intervals. Do they ever differ?

In [ ]:
# GIVEN
rng = np.random.default_rng(SEED + 3); agree = 0; rips_differs = 0
for _ in range(30):
    P = rng.uniform(size=(7, 2)); c = loop_intervals(7, *cech_complex(P), 0.8); a = loop_intervals(7, *alpha_complex(P), 0.8); r_ = loop_intervals(7, *rips_complex(P), 0.8)
    agree += (len(c) == len(a) and all(abs(x - y) < 1e-7 for u, v in zip(c, a) for x, y in zip(u, v)))
    rips_differs += not (len(c) == len(r_) and all(abs(x - y) < 1e-7 for u, v in zip(c, r_) for x, y in zip(u, v)))
print(f"Cech and Alpha have the same loop intervals in {agree} of 30 clouds; Rips differs from Cech in {rips_differs} of 30")
vals_ok = all(alpha_complex(P)[1][t] >= cech_triangle_value(*P[list(t)]) - 1e-12 for P in [rng.uniform(size=(8, 2)) for _ in range(10)] for t in alpha_complex(P)[1])
print("every Alpha triangle enters no earlier than in the Cech complex (Alpha(r) is contained in Cech(r)):", vals_ok)

In [ ]:
# CHECK: Part D
assert agree == 30, "Cech and Alpha always agree on the loops"
assert vals_ok, "Alpha(r) is a subcomplex of Cech(r)"
assert rips_differs > 0, "Rips does differ from Cech on some clouds"
print("Exercise 3 passed.")

*Discuss:* in which of the four clouds could you have predicted the three answers by hand? What does the hexagon show about the Rips complex that the square did not?

## Exercise 4 — Delaunay and the cost of a complex

Why use the Alpha complex at all? Because it lives inside the Delaunay triangulation, which is small. You write:

1. `empty_circle_violations(P, simplices)`: the number of (point, triangle) pairs in which a point lies **strictly inside** the circumcircle of the triangle (the points of the triangle itself do not count; use a small tolerance, `R - 1e-9`). For a Delaunay triangulation it must be 0.
2. `euler_characteristic(n, simplices)`: $V-E+T$, where the edges are the distinct edges of the triangles. A triangulated disk has the value 1.
3. `delaunay_sizes_2d(P)`: the number of triangles and the number of edges **per point**; and `delaunay_sizes_3d(P)`: the number of tetrahedra per point.
4. `count_by_dimension(simplex_tree)`: the number of simplices of each dimension of a `gudhi` simplex tree (`get_simplices()` yields `(simplex, filtration)` pairs).

In [ ]:
def empty_circle_violations(P, simplices):
    """Number of (point, triangle) pairs with the point strictly inside the circumcircle of the triangle."""
    # TODO
    raise NotImplementedError

def euler_characteristic(n, simplices):
    """V - E + T of a triangulation given by its triangles."""
    # TODO
    raise NotImplementedError

def delaunay_sizes_2d(P):
    """(triangles per point, edges per point) of the Delaunay triangulation of planar points."""
    # TODO
    raise NotImplementedError

def delaunay_sizes_3d(P):
    """Tetrahedra per point of the Delaunay triangulation of points in space."""
    # TODO
    raise NotImplementedError

def count_by_dimension(st):
    """Number of simplices of each dimension in a gudhi simplex tree."""
    # TODO
    raise NotImplementedError

**Part A.** Delaunay triangulations of random points: the empty-circle property, the Euler characteristic, and how many simplices per point.

In [ ]:
# GIVEN
rng = np.random.default_rng(SEED + 10)
P2 = rng.uniform(size=(300, 2)); T2 = Delaunay(P2).simplices; viol = empty_circle_violations(P2, T2); chi = euler_characteristic(len(P2), T2)
tri_pp, edge_pp = delaunay_sizes_2d(rng.uniform(size=(2000, 2))); tet_pp = delaunay_sizes_3d(rng.uniform(size=(3000, 3)))
print(f"empty-circle violations: {viol}; V - E + T = {chi}; plane: {tri_pp:.2f} triangles and {edge_pp:.2f} edges per point; space: {tet_pp:.2f} tetrahedra per point")

In [ ]:
# CHECK: Part A
assert viol == 0, "a Delaunay triangulation has an empty circumcircle for every triangle"
assert chi == 1, "a triangulated disk has Euler characteristic 1"
assert 1.9 < tri_pp < 2.0 and 2.9 < edge_pp < 3.0, "about 2 triangles and 3 edges per point"
assert 6.2 < tet_pp < 6.8, "about 6.5 tetrahedra per point"
assert empty_circle_violations(P2[:10], Delaunay(P2[:10]).simplices) == 0
assert empty_circle_violations(np.array([[0, 0], [1, 0], [0.5, 0.8], [0.5, 0.3]]), np.array([[0, 1, 2]])) == 1, "the fourth point is inside the circle through the first three"
print("Exercise 4, Part A passed.")

**Part B.** The cost: 400 points on a torus, as an Alpha complex and as Rips complexes of three sizes (up to dimension 3, so the Rips counts are lower bounds). The Rips parameter below is the **edge length** $\varepsilon=2r$ (gudhi's convention).

In [ ]:
# GIVEN
def torus_points(n, rng, R=2.0, r=1.0):
    out = []
    while len(out) < n:
        u, v = rng.uniform(0, 2 * np.pi, 2)
        if rng.uniform(0, R + r) < R + r * np.cos(v): out.append([(R + r * np.cos(v)) * np.cos(u), (R + r * np.cos(v)) * np.sin(u), r * np.sin(v)])
    return np.array(out)
Tor = torus_points(400, np.random.default_rng(SEED + 11))
alpha_counts = count_by_dimension(gudhi.AlphaComplex(points=Tor).create_simplex_tree())
rips_counts = {eps: count_by_dimension(gudhi.RipsComplex(points=Tor, max_edge_length=eps).create_simplex_tree(max_dimension=3)) for eps in (0.7, 1.1, 1.5)}
print("alpha:", alpha_counts, "total", sum(alpha_counts))
for eps, c in rips_counts.items(): print(f"Rips, edge length {eps}:", c, "total", sum(c))

In [ ]:
# CHECK: Part B
totals = {eps: sum(c) for eps, c in rips_counts.items()}; ta = sum(alpha_counts)
assert alpha_counts[0] == 400 and len(alpha_counts) == 4, "the alpha complex of points in space has dimension at most 3"
assert totals[0.7] < ta < totals[1.1] < totals[1.5], "Rips is smaller than Alpha at small scales and explodes at larger ones"
assert totals[1.5] > 10 * ta, "at a scale where the loops are present, Rips is more than ten times larger"
print("Exercise 4 passed.")

*Discuss:* Alpha has about $2N$ triangles in the plane and $6.5N$ tetrahedra in space no matter what the scale is. Why can the Rips complex not promise anything like that? And why can we not simply use Alpha for everything (think of a distance matrix of molecules)?

## Exercise 5 — Homotopy and knots (optional, take-home)

A homotopy between two closed curves may pass the curve through itself; an isotopy may not. Check it numerically on the straight-line homotopy $H_s=(1-s)K_0+sK_1$ from a trefoil to a circle.

You write:

1. `straight_line_homotopy(A, B, s)`: the curve $(1-s)A+sB$ for two curves given as arrays of $N$ points.
2. `min_far_distance(P, k)`: the smallest distance between two points of the closed curve `P` that are at least `k` steps apart along it (in either direction): the closest two *distant* strands come.
3. `alexander_determinant(coeffs)`: $|\Delta(-1)|=\big|\sum_j (-1)^j c_j\big|$ for a polynomial with coefficient list `coeffs` $(c_0,c_1,\dots)$.
4. `find_collisions(dets)`: the pairs of knots with the same determinant.

In [ ]:
def straight_line_homotopy(A, B, s):
    """The curve (1 - s) A + s B."""
    # TODO
    raise NotImplementedError

def min_far_distance(P, k):
    """Smallest distance between points of a closed curve that are at least k steps apart along it."""
    # TODO
    raise NotImplementedError

def alexander_determinant(coeffs):
    """|Delta(-1)| of a polynomial given by its coefficient list."""
    # TODO
    raise NotImplementedError

def find_collisions(dets):
    """Pairs of names with the same determinant."""
    # TODO
    raise NotImplementedError

**Part A.** A trefoil, a circle, and a bent circle (an unknot); 240 points each.

In [ ]:
# GIVEN
NK = 240; t = np.linspace(0, 2 * np.pi, NK, endpoint=False)
trefoil = np.column_stack([np.sin(t) + 2 * np.sin(2 * t), np.cos(t) - 2 * np.cos(2 * t), -np.sin(3 * t)]) / 2.2
circle3 = np.column_stack([np.cos(t), np.sin(t), 0 * t]); bent = np.column_stack([np.cos(t), np.sin(t), 0.6 * np.sin(2 * t)])
ss = np.linspace(0, 1, 101); k = NK // 12
d_tc = np.array([min_far_distance(straight_line_homotopy(trefoil, circle3, s), k) for s in ss]); d_cb = np.array([min_far_distance(straight_line_homotopy(circle3, bent, s), k) for s in ss])
print(f"trefoil to circle: smallest distance {d_tc.min():.4f} at s = {ss[d_tc.argmin()]:.2f} (at the ends: {d_tc[0]:.3f} and {d_tc[-1]:.3f})")
print(f"circle to bent circle (two unknots): smallest distance {d_cb.min():.3f}")
fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(ss, d_tc, color=AMBER, label="trefoil to circle"); ax.plot(ss, d_cb, color=TEAL, label="circle to bent circle"); ax.set_xlabel("s"); ax.set_ylabel("closest distant strands"); ax.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part A
assert d_tc.min() < 0.05, "on the way from the trefoil to the circle the strands touch: the curve passes through itself"
assert d_tc[0] > 0.4 and d_tc[-1] > 0.4, "both end curves are embedded: their strands are far apart"
assert 0.25 < ss[d_tc.argmin()] < 0.45, "the crossing happens in the first half of the homotopy"
assert d_cb.min() > 0.4, "a homotopy between two unknots need not pass through itself"
print("Exercise 5, Part A passed.")

**Part B.** Alexander polynomials from the knot table (coefficient lists, up to units), and a single number made from them.

In [ ]:
# GIVEN
alexander = {"3_1": [1, -1, 1], "4_1": [1, -3, 1], "5_1": [1, -1, 1, -1, 1], "5_2": [2, -3, 2], "6_1": [2, -5, 2], "7_1": [1, -1, 1, -1, 1, -1, 1]}
dets = {k: alexander_determinant(v) for k, v in alexander.items()}; coll = find_collisions(dets)
print("determinants:", dets); print("knots with the same determinant:", coll)
print("the polynomials of those knots are different:", [(alexander[a], alexander[b]) for a, b in coll])

In [ ]:
# CHECK: Part B
assert dets == {"3_1": 3, "4_1": 5, "5_1": 5, "5_2": 7, "6_1": 9, "7_1": 7}, dets
assert sorted(coll) == [("4_1", "5_1"), ("5_2", "7_1")], coll
assert all(alexander[a] != alexander[b] for a, b in coll), "the polynomial still separates them"
print("Exercise 5 passed.")

*Discuss:* what did the straight-line homotopy allow that an isotopy does not? A single invariant (the determinant) already fails to separate five of these six knots; the polynomial separates them, but there are knots it cannot separate either. What does that say about classifying knots by one number?

## What to remember

* Distances are blind to connectedness: a gap of 0.02 rad is almost invisible to W₁, and the loop is already cut.
* A random sample leaves gaps; for a gap of 0.5 rad you need about 92 points to be below a 5% chance, and about twice as many each time the gap halves.
* Čech, Rips and Alpha are three rules on the same points. Čech and Alpha agree on the loops; Rips can close a loop early (hexagon) or never open it (equilateral triangle). Libraries differ in units: edge length for ripser, squared radius for gudhi.
* The Alpha complex lives in the Delaunay triangulation, which has about $2N$ triangles in the plane and $6.5N$ tetrahedra in space; Rips explodes with the scale.
* A homotopy may cross a curve through itself, an isotopy may not; one number does not classify knots.